# 07 — Model Playground
## bd_replica_crm · IDE analítico de modelos dentro del DW

Este notebook permite configurar desde una sola celda:

```python
TARGET = "minuta_60d"
MODEL = "Logit"
FEATURE_SET = "commercial"
SEGMENT = None
```

y construir automáticamente:

- dataset;
- filtros;
- train/test temporal;
- métricas;
- parámetros;
- efectos marginales cuando aplican;
- lift / calibration para clasificación;
- errores;
- comparación entre modelos;
- interpretación ejecutiva;
- gates de calidad.

Modelos soportados inicialmente:

- OLS
- Logit
- Poisson
- LogisticRegression
- RandomForest
- GradientBoosting

El notebook está diseñado para crecer después hacia Survival, Cox, DiD, RDD, Uplift y Causal Forest.


## 0. Setup


In [2]:
from __future__ import annotations

import sys
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
!pip install matplotlib
import matplotlib.pyplot as plt

import statsmodels.api as sm
import statsmodels.formula.api as smf

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    mean_squared_error,
    mean_absolute_error,
)
from sklearn.model_selection import train_test_split

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / "pyproject.toml").exists() else cwd.parent

if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Ejecuta este notebook dentro de bd_replica_crm.")

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from replica_cygnus.settings import load_settings
from replica_cygnus.connections import connect_postgres

settings = load_settings(PROJECT_ROOT)
conn = connect_postgres(settings)

pd.set_option("display.max_columns", 180)
pd.set_option("display.max_rows", 180)
pd.set_option("display.width", 260)

def sql_df(sql, params=None):
    return pd.read_sql_query(sql, conn, params=params)

print("DB:", settings.postgres.database)
print("Inicio:", datetime.now().astimezone().isoformat(timespec="seconds"))



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached matplotlib-3.11.1-cp312-cp312-win_amd64.whl.metadata (80 kB)
  Using cached contourpy-1.3.3-cp312-cp312-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pillow-12.3.0-cp312-cp312-win_amd64.whl.metadata (9.3 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
Using cached matplotlib-3.11.1-cp312-cp312-win_amd64.whl (9.3 MB)
Using cached contourpy-1.3.3-cp312-cp312-win_amd64.whl (226 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   --------------------------------- ------ 2.1/2.5 MB 11.8 MB/s eta 0:00:01
   ---------------------------------------- 2.5/2.5 MB 10.1 MB/s eta 0:00:00
Using cached pillow-12.3.0-cp312-cp312-win_amd64.whl (7.2 MB)
Using cached pyparsing-3.3.2-py3-none-any.whl (122 kB)
DB: medallio_dw
Inicio: 2026-09-07T14:52:51-05:00


## 1. Configuración principal


In [3]:
TARGET = "minuta_60d"

MODEL = "Logit"
# Opciones:
# OLS
# Logit
# Poisson
# LogisticRegression
# RandomForest
# GradientBoosting

FEATURE_SET = "commercial"
# commercial
# behavior
# historical_rates
# compact
# all

SEGMENT = None
# Ejemplos:
# SEGMENT = {"codigo_proyecto": "Matera"}
# SEGMENT = {"asesor": "usuario_x"}
# SEGMENT = {"canal": "Meta"}

TEST_FRACTION = 0.20
RANDOM_STATE = 42

print("TARGET =", TARGET)
print("MODEL =", MODEL)
print("FEATURE_SET =", FEATURE_SET)
print("SEGMENT =", SEGMENT)


TARGET = minuta_60d
MODEL = Logit
FEATURE_SET = commercial
SEGMENT = None


## 2. Catálogo de features


In [4]:
FEATURE_SETS = {
    "compact": [
        "hour_of_day",
        "day_of_week",
        "is_weekend",
        "client_prior_assignments_90d",
        "days_since_previous_assignment",
    ],
    "behavior": [
        "hour_of_day",
        "day_of_week",
        "is_weekend",
        "client_prior_assignments_90d",
        "days_since_previous_assignment",
        "project_leads_90d",
        "advisor_leads_90d",
    ],
    "historical_rates": [
        "project_sep_rate_90d",
        "project_minuta_rate_180d",
        "advisor_sep_rate_90d",
        "advisor_minuta_rate_180d",
        "global_sep_rate_90d",
        "global_minuta_rate_180d",
    ],
    "commercial": [
        "codigo_proyecto",
        "asesor",
        "canal",
        "medio",
        "hour_of_day",
        "day_of_week",
        "is_weekend",
        "client_prior_assignments_90d",
        "days_since_previous_assignment",
        "project_leads_90d",
        "project_sep_rate_90d",
        "project_minuta_rate_180d",
        "advisor_leads_90d",
        "advisor_sep_rate_90d",
        "advisor_minuta_rate_180d",
        "global_sep_rate_90d",
        "global_minuta_rate_180d",
    ],
}

FEATURE_SETS["all"] = FEATURE_SETS["commercial"]

features = FEATURE_SETS[FEATURE_SET]
features


['codigo_proyecto',
 'asesor',
 'canal',
 'medio',
 'hour_of_day',
 'day_of_week',
 'is_weekend',
 'client_prior_assignments_90d',
 'days_since_previous_assignment',
 'project_leads_90d',
 'project_sep_rate_90d',
 'project_minuta_rate_180d',
 'advisor_leads_90d',
 'advisor_sep_rate_90d',
 'advisor_minuta_rate_180d',
 'global_sep_rate_90d',
 'global_minuta_rate_180d']

## 3. Dataset real


In [5]:
cols = [
    "evidence_key",
    "lead_id",
    "decision_at",
    "codigo_proyecto",
    "asesor",
    "canal",
    "medio",
    "hour_of_day",
    "day_of_week",
    "is_weekend",
    "client_prior_assignments_90d",
    "days_since_previous_assignment",
    "project_leads_90d",
    "project_sep_rate_90d",
    "project_minuta_rate_180d",
    "advisor_leads_90d",
    "advisor_sep_rate_90d",
    "advisor_minuta_rate_180d",
    "global_sep_rate_90d",
    "global_minuta_rate_180d",
    "separacion_14d",
    "minuta_60d",
]

data = sql_df(
    "SELECT " + ", ".join(cols) + " FROM features.lead_evidence"
)

data["decision_at"] = pd.to_datetime(
    data["decision_at"],
    utc=True,
    errors="coerce"
)

print("rows:", len(data))
display(data.head())


C:\Users\dinat\AppData\Local\Temp\ipykernel_19156\286139807.py:51: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn, params=params)


rows: 206043


,evidence_key,lead_id,decision_at,codigo_proyecto,asesor,canal,medio,hour_of_day,day_of_week,is_weekend,client_prior_assignments_90d,days_since_previous_assignment,project_leads_90d,project_sep_rate_90d,project_minuta_rate_180d,advisor_leads_90d,advisor_sep_rate_90d,advisor_minuta_rate_180d,global_sep_rate_90d,global_minuta_rate_180d,separacion_14d,minuta_60d
0,f994e4aa385df9f82790804055fd4556,122839,2024-07-31 15:27:45.230310+00:00,GY,None,None,3 días de locura inmobiliaria,10,3,0,None,None,None,None,None,None,None,None,None,None,0.0,0.0
1,16503383413d5776f724ccd18ceeb415,122840,2024-07-17 17:04:44.809686+00:00,GY,None,None,3 días de locura inmobiliaria,12,3,0,None,None,None,None,None,None,None,None,None,None,0.0,0.0
2,e1682806e4b5bef9ad763360af5b0e05,122845,2024-07-09 20:59:49.638926+00:00,EEUU,None,None,3 días de locura inmobiliaria,15,2,0,None,None,None,None,None,None,None,None,None,None,0.0,0.0
3,f6a2ac8b5f26fb4c966d0727fd7c6b91,122846,2024-07-30 02:30:44.796950+00:00,FX,None,None,3 días de locura inmobiliaria,21,1,0,None,None,None,None,None,None,None,None,None,None,0.0,0.0
4,c51a3999e74e7d2ff62053ec8b8bea96,122848,2024-07-09 20:59:49.638926+00:00,FX,None,None,3 días de locura inmobiliaria,15,2,0,None,None,None,None,None,None,None,None,None,None,0.0,0.0


## 4. Aplicar segmento


In [6]:
model_data = data.copy()

if SEGMENT:
    for col, value in SEGMENT.items():
        if col not in model_data.columns:
            raise KeyError(f"Segment column no existe: {col}")
        model_data = model_data[model_data[col].eq(value)]

model_data = model_data[
    model_data[TARGET].notna()
].copy()

print("Rows después de segmentación y target maduro:", len(model_data))
print("Target mean:", model_data[TARGET].mean())


Rows después de segmentación y target maduro: 198809
Target mean: 0.0017655136336886157


## 5. Calidad de features


In [7]:
quality = pd.DataFrame([
    {
        "feature": c,
        "dtype": str(model_data[c].dtype),
        "null_pct": model_data[c].isna().mean(),
        "nunique": model_data[c].nunique(dropna=True),
    }
    for c in features
]).sort_values(["null_pct", "nunique"], ascending=[False, True])

quality


,feature,dtype,null_pct,nunique
1,asesor,object,1.000000,0
2,canal,object,1.000000,0
7,client_prior_assignments_90d,object,1.000000,0
8,days_since_previous_assignment,object,1.000000,0
9,project_leads_90d,object,1.000000,0
10,project_sep_rate_90d,object,1.000000,0
11,project_minuta_rate_180d,object,1.000000,0
12,advisor_leads_90d,object,1.000000,0
13,advisor_sep_rate_90d,object,1.000000,0
14,advisor_minuta_rate_180d,object,1.000000,0


## 6. Split temporal


In [8]:
model_data = model_data.sort_values(
    ["decision_at", "evidence_key"]
).reset_index(drop=True)

split_idx = int(len(model_data) * (1 - TEST_FRACTION))

train = model_data.iloc[:split_idx].copy()
test = model_data.iloc[split_idx:].copy()

print("TRAIN:", len(train), train["decision_at"].min(), "→", train["decision_at"].max())
print("TEST :", len(test), test["decision_at"].min(), "→", test["decision_at"].max())


TRAIN: 159047 2019-10-02 23:08:14.794272+00:00 → 2025-10-31 22:59:00.467301+00:00
TEST : 39762 2025-10-31 23:25:04.953167+00:00 → 2026-07-10 04:46:05.529447+00:00


## 7. Separar numéricas y categóricas


In [9]:
categorical_features = [
    c for c in features
    if model_data[c].dtype == "object"
    or str(model_data[c].dtype).startswith("string")
]

numeric_features = [
    c for c in features
    if c not in categorical_features
]

print("Categorical:", categorical_features)
print("Numeric:", numeric_features)


Categorical: ['codigo_proyecto', 'asesor', 'canal', 'medio', 'client_prior_assignments_90d', 'days_since_previous_assignment', 'project_leads_90d', 'project_sep_rate_90d', 'project_minuta_rate_180d', 'advisor_leads_90d', 'advisor_sep_rate_90d', 'advisor_minuta_rate_180d', 'global_sep_rate_90d', 'global_minuta_rate_180d']
Numeric: ['hour_of_day', 'day_of_week', 'is_weekend']


## 8. Statsmodels: OLS / Logit


In [10]:
statsmodels_result = None

safe_numeric = [
    c for c in numeric_features
    if train[c].notna().sum() > 0
]

if MODEL in {"OLS", "Logit"}:
    formula_features = safe_numeric.copy()

    if not formula_features:
        print("No hay features numéricas suficientes.")
    else:
        formula = TARGET + " ~ " + " + ".join(formula_features)

        sm_data = train[
            [TARGET] + formula_features
        ].dropna()

        print("Formula:", formula)
        print("n:", len(sm_data))

        if MODEL == "OLS" and len(sm_data) >= 30:
            statsmodels_result = smf.ols(
                formula,
                data=sm_data
            ).fit(cov_type="HC3")
            print(statsmodels_result.summary())

        elif (
            MODEL == "Logit"
            and len(sm_data) >= 30
            and sm_data[TARGET].nunique() == 2
        ):
            statsmodels_result = smf.logit(
                formula,
                data=sm_data
            ).fit(disp=False)
            print(statsmodels_result.summary())
            print("\nEFECTOS MARGINALES")
            print(statsmodels_result.get_margeff().summary())


Formula: minuta_60d ~ hour_of_day + day_of_week + is_weekend
n: 159047
                           Logit Regression Results                           
Dep. Variable:             minuta_60d   No. Observations:               159047
Model:                          Logit   Df Residuals:                   159043
Method:                           MLE   Df Model:                            3
Date:                Mon, 07 Sep 2026   Pseudo R-squ.:                0.007347
Time:                        14:52:57   Log-Likelihood:                -1039.4
converged:                       True   LL-Null:                       -1047.1
Covariance Type:            nonrobust   LLR p-value:                  0.001515
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept      -8.0355      0.397    -20.232      0.000      -8.814      -7.257
hour_of_day     0.0407      0.020      2.024      0.043  

## 9. Poisson


In [11]:
poisson_result = None

if MODEL == "Poisson":
    poisson_df = (
        model_data.assign(
            decision_date=model_data["decision_at"].dt.date
        )
        .groupby(
            ["decision_date", "codigo_proyecto"],
            dropna=False
        )
        .agg(
            events=(TARGET, "sum"),
            exposures=("evidence_key", "count"),
            avg_project_sep_rate=("project_sep_rate_90d", "mean"),
            avg_project_minuta_rate=("project_minuta_rate_180d", "mean"),
        )
        .reset_index()
    )

    poisson_df = poisson_df.dropna(
        subset=[
            "events",
            "exposures",
            "avg_project_sep_rate",
            "avg_project_minuta_rate",
        ]
    )

    poisson_df = poisson_df[
        poisson_df["exposures"] > 0
    ].copy()

    poisson_df["log_exposure"] = np.log(
        poisson_df["exposures"]
    )

    print("Poisson rows:", len(poisson_df))

    if len(poisson_df) >= 30:
        poisson_result = smf.glm(
            formula=(
                "events ~ avg_project_sep_rate "
                "+ avg_project_minuta_rate"
            ),
            data=poisson_df,
            family=sm.families.Poisson(),
            offset=poisson_df["log_exposure"],
        ).fit()

        print(poisson_result.summary())
    else:
        print("PENDING: muestra insuficiente.")


## 10. Pipeline sklearn


In [12]:
sk_model = None
sk_metrics = None
test_predictions = None

if MODEL in {
    "LogisticRegression",
    "RandomForest",
    "GradientBoosting"
}:
    X_train = train[features]
    y_train = train[TARGET].astype(int)

    X_test = test[features]
    y_test = test[TARGET].astype(int)

    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore")),
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_pipe, numeric_features),
        ("cat", categorical_pipe, categorical_features),
    ])

    if MODEL == "LogisticRegression":
        estimator = LogisticRegression(
            max_iter=1500,
            random_state=RANDOM_STATE
        )

    elif MODEL == "RandomForest":
        estimator = RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=10,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )

    else:
        estimator = GradientBoostingClassifier(
            random_state=RANDOM_STATE
        )

    sk_model = Pipeline([
        ("prep", preprocessor),
        ("model", estimator),
    ])

    sk_model.fit(X_train, y_train)

    prob = sk_model.predict_proba(X_test)[:, 1]

    sk_metrics = {
        "model": MODEL,
        "n_test": len(y_test),
        "base_rate": y_test.mean(),
        "roc_auc": roc_auc_score(y_test, prob)
            if y_test.nunique() > 1 else np.nan,
        "average_precision": average_precision_score(
            y_test, prob
        ),
        "brier": brier_score_loss(y_test, prob),
    }

    test_predictions = test[
        [
            "evidence_key",
            "decision_at",
            "codigo_proyecto",
            TARGET
        ]
    ].copy()

    test_predictions["prob"] = prob

    display(pd.DataFrame([sk_metrics]))


## 11. Lift Top-K


In [13]:
if test_predictions is not None:
    ranked = test_predictions.sort_values(
        "prob",
        ascending=False
    ).reset_index(drop=True)

    total_pos = ranked[TARGET].sum()

    rows = []

    for frac in [.05, .10, .20, .30, .50]:
        n = max(1, int(len(ranked) * frac))
        top = ranked.head(n)

        rows.append({
            "top_fraction": frac,
            "n": n,
            "target_rate": top[TARGET].mean(),
            "base_rate": ranked[TARGET].mean(),
            "lift": (
                top[TARGET].mean()
                / ranked[TARGET].mean()
                if ranked[TARGET].mean() > 0
                else np.nan
            ),
            "positives_captured": (
                top[TARGET].sum() / total_pos
                if total_pos > 0 else np.nan
            )
        })

    lift_table = pd.DataFrame(rows)
    display(lift_table)


## 12. Calibration por deciles


In [14]:
if test_predictions is not None:
    cal = test_predictions[[TARGET, "prob"]].copy()

    cal["decile"] = pd.qcut(
        cal["prob"],
        q=10,
        duplicates="drop"
    )

    calibration = (
        cal.groupby("decile", observed=True)
        .agg(
            n=(TARGET, "size"),
            predicted=("prob", "mean"),
            observed=(TARGET, "mean"),
        )
        .reset_index()
    )

    calibration["gap_pp"] = (
        calibration["predicted"]
        - calibration["observed"]
    ) * 100

    display(calibration)


## 13. Errores extremos


In [15]:
if test_predictions is not None:
    false_positive = (
        test_predictions[
            test_predictions[TARGET].eq(0)
        ]
        .sort_values("prob", ascending=False)
        .head(20)
    )

    false_negative = (
        test_predictions[
            test_predictions[TARGET].eq(1)
        ]
        .sort_values("prob", ascending=True)
        .head(20)
    )

    print("FALSOS POSITIVOS EXTREMOS")
    display(false_positive)

    print("FALSOS NEGATIVOS EXTREMOS")
    display(false_negative)


## 14. Parámetros interpretables


In [16]:
if statsmodels_result is not None:
    ci = statsmodels_result.conf_int()

    parameter_table = pd.DataFrame({
        "parameter": statsmodels_result.params.index,
        "coef": statsmodels_result.params.values,
        "std_err": statsmodels_result.bse.values,
        "p_value": statsmodels_result.pvalues.values,
        "ci_low": ci.iloc[:, 0].values,
        "ci_high": ci.iloc[:, 1].values,
    })

    display(parameter_table)


,parameter,coef,std_err,p_value,ci_low,ci_high
0,Intercept,-8.035457,0.397157,5.071131e-91,-8.813870,-7.257044
1,hour_of_day,0.040663,0.020088,4.294855e-02,0.001291,0.080036
2,day_of_week,0.041987,0.075345,5.773519e-01,-0.105688,0.189661
3,is_weekend,0.477166,0.319156,1.348921e-01,-0.148369,1.102700


## 15. Comparador rápido de modelos


In [17]:
COMPARE_MODELS = False

comparison_rows = []

if COMPARE_MODELS:
    candidate_models = [
        "LogisticRegression",
        "RandomForest",
        "GradientBoosting",
    ]

    X_train = train[features]
    y_train = train[TARGET].astype(int)
    X_test = test[features]
    y_test = test[TARGET].astype(int)

    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore")),
    ])

    prep = ColumnTransformer([
        ("num", numeric_pipe, numeric_features),
        ("cat", categorical_pipe, categorical_features),
    ])

    estimators = {
        "LogisticRegression": LogisticRegression(
            max_iter=1500,
            random_state=RANDOM_STATE
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=10,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "GradientBoosting": GradientBoostingClassifier(
            random_state=RANDOM_STATE
        ),
    }

    for name, estimator in estimators.items():
        p = Pipeline([
            ("prep", prep),
            ("model", estimator),
        ])

        p.fit(X_train, y_train)
        prob = p.predict_proba(X_test)[:, 1]

        comparison_rows.append({
            "model": name,
            "roc_auc": roc_auc_score(y_test, prob)
                if y_test.nunique() > 1 else np.nan,
            "average_precision": average_precision_score(
                y_test, prob
            ),
            "brier": brier_score_loss(y_test, prob),
        })

    comparison = pd.DataFrame(
        comparison_rows
    ).sort_values(
        "average_precision",
        ascending=False
    )

    display(comparison)
else:
    print("COMPARE_MODELS=False")


COMPARE_MODELS=False


## 16. Gate automático


In [18]:
gates = []

def gate(name, passed, detail):
    gates.append({
        "gate": name,
        "status": "PASS" if passed else "FAIL",
        "detail": detail,
    })

gate(
    "Muestra suficiente",
    len(model_data) >= 100,
    f"n={len(model_data):,}"
)

gate(
    "Target tiene variación",
    model_data[TARGET].nunique() > 1,
    f"classes={model_data[TARGET].nunique()}"
)

gate(
    "Split temporal válido",
    len(train) > 0 and len(test) > 0,
    f"train={len(train):,}, test={len(test):,}"
)

if sk_metrics is not None:
    gate(
        "AUC > azar",
        pd.notna(sk_metrics["roc_auc"])
        and sk_metrics["roc_auc"] > .50,
        f"AUC={sk_metrics['roc_auc']:.3f}"
    )

gate_table = pd.DataFrame(gates)
gate_table


,gate,status,detail
0,Muestra suficiente,PASS,"n=198,809"
1,Target tiene variación,PASS,classes=2
2,Split temporal válido,PASS,"train=159,047, test=39,762"


## 17. Interpretación ejecutiva automática


In [19]:
print("=== MODEL PLAYGROUND ===")
print("Target:", TARGET)
print("Model:", MODEL)
print("Feature set:", FEATURE_SET)
print("Segment:", SEGMENT)
print("Rows:", len(model_data))

if sk_metrics is not None:
    print(
        f"AUC={sk_metrics['roc_auc']:.3f} | "
        f"AP={sk_metrics['average_precision']:.3f} | "
        f"Brier={sk_metrics['brier']:.3f}"
    )

if "lift_table" in globals() and len(lift_table):
    top20 = lift_table[
        np.isclose(
            lift_table["top_fraction"],
            .20
        )
    ]

    if len(top20):
        r = top20.iloc[0]
        print(
            f"Top 20% lift={r['lift']:.2f}x | "
            f"captura={r['positives_captured']:.1%}"
        )


=== MODEL PLAYGROUND ===
Target: minuta_60d
Model: Logit
Feature set: commercial
Segment: None
Rows: 198809


## 18. Export opcional


In [20]:
EXPORT = False

if EXPORT:
    out = PROJECT_ROOT / "reports" / "model_playground"
    out.mkdir(parents=True, exist_ok=True)

    gate_table.to_csv(
        out / "latest_gates.csv",
        index=False
    )

    if test_predictions is not None:
        test_predictions.to_csv(
            out / "latest_predictions.csv",
            index=False
        )

    print("Export:", out)
else:
    print("EXPORT=False")


EXPORT=False


## 19. Próximas extensiones


El siguiente nivel del playground puede incorporar:

```text
Survival / Cox
Panel Fixed Effects
DiD
RDD
Synthetic Control
IPW / AIPW
Double Machine Learning
Causal Forest
Uplift
Bayesian Hierarchical
```

La filosofía es mantener una sola interfaz de configuración y cambiar el motor analítico por debajo.


In [21]:
conn.close(); print("Conexión cerrada.")


Conexión cerrada.
